# OpenFE Benchmark/FEP+ via Boltz 3D Affinity API

This notebook runs the full OpenFE binding-affinity benchmark (56 proteins, ~1054 ligands) through the Boltz2 async prediction API.

**Steps:**
1. Setup and connect to the API
2. Upload pre-computed MSA files
3. Submit all protein-ligand prediction jobs
4. Monitor completion and collect results
5. Merge with experimental ground truth
6. Compute correlation metrics

## 1. Setup

In [ ]:
import glob
import os
import copy
import json
import string
import time

import pandas as pd
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from scipy.stats import spearmanr, pearsonr

from boltz_api_client import (
    AsyncApiClient,
    ProteinInput,
    LigandInput,
    load_pdb_file,
    load_molblock_file,
)

In [ ]:
API_URL = os.getenv("API_URL", "BOLTZ_API_URL")
API_KEY = os.getenv("API_KEY", "YOUR_API_KEY")

client = AsyncApiClient(API_URL, api_key=API_KEY)

In [ ]:
health = client.health_check()
print(f"API Status: {health['status']}")

quota = client.get_quota()
print(f"Quota: {quota.quota_used}/{quota.quota_max} used, {quota.quota_remaining} remaining")

## 2. Upload MSA Files

Upload all pre-computed MSA CSV files (one per protein chain) to S3. The returned filenames are used later when building `ProteinInput` objects.

In [ ]:
msa_files = glob.glob("data/msa/*.csv")

for msa_file in msa_files:
    client.upload_msa(msa_file)

print(f"\nUploaded {len(msa_files)} MSA files")

## 3. Helper: Extract Chain Sequences from PDB

Each protein chain needs its own MSA file. The naming convention is `{protein_name}_{chain_id}.csv`. We use BioPython to extract chain IDs and amino-acid sequences from the cleaned PDB, then map identical sequences to the same MSA file.

In [ ]:
def get_chains_and_sequences(pdb_file):
    """Extract standard-residue sequences per chain from a PDB file.

    Returns
    -------
    tuple[list[str], list[str]]
        (chain_ids, sequences)
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)

    chains_residues = {}
    for model in structure:
        for chain in model:
            chains_residues[chain.id] = [
                residue for residue in chain if residue.id[0] == " "
            ]

    result = {}
    for chain_id, residues in chains_residues.items():
        sequence = "".join(seq1(r.get_resname()) for r in residues)
        if sequence:
            result[chain_id] = sequence

    return list(result.keys()), list(result.values())

## 4. Submit All Prediction Jobs

Loop over all 56 protein folders in `openfe_benchmark/`. For each protein:
1. Load the cleaned PDB and determine which MSA files to attach (one per chain)
2. Load all ligand SDF files
3. Submit the job via `create_request`
4. Save request IDs to a JSON file for resumability

In [ ]:
folders_prot = sorted(glob.glob("data/benchmark/*/"))
print(f"Found {len(folders_prot)} protein targets")

In [ ]:
RUN_NAME = "run-benchmark"  # change for each new run
HISTORY_FILE = f"history_requests_{RUN_NAME}.json"

requests = {}
for folder in tqdm(folders_prot, desc="Submitting jobs"):
    protein_file = os.path.join(folder, "protein_clean.pdb")
    protein_name = os.path.basename(folder.rstrip("/"))
    protein_pdb = load_pdb_file(protein_file)

    # Build MSA filenames: one per chain, reusing the same file for identical sequences
    _, sequences = get_chains_and_sequences(protein_file)
    chain_ids = [string.ascii_uppercase[i] for i in range(len(sequences))]
    sequence_to_chain = {}
    msa_list = []
    for seq, cid in zip(sequences, chain_ids):
        if seq not in sequence_to_chain:
            sequence_to_chain[seq] = cid
        msa_list.append(f"{protein_name}_{sequence_to_chain[seq]}.csv")

    protein = ProteinInput(
        name=protein_name, pdb_block=protein_pdb, msa_filenames=msa_list
    )

    # Load ligands
    ligands = []
    for ligand_file in glob.glob(os.path.join(folder, "ligands/*.sdf")):
        molblock = load_molblock_file(ligand_file)
        ligand_name = os.path.basename(ligand_file).replace(".sdf", "").replace(".", "_")
        ligands.append(LigandInput(name=ligand_name, molblock=molblock))

    # Submit
    job_name = f"{protein_name}-benchmark-openfe-{RUN_NAME}"
    response = client.create_request(
        job_name=job_name,
        protein=protein,
        ligands=ligands,
    )
    requests[job_name] = {
        "parent_request_id": response["parent_request_id"],
        "nb_ligands": len(ligands),
    }

    # Save incrementally for resumability
    with open(HISTORY_FILE, "w") as f:
        json.dump(requests, f)

total_ligands = sum(v["nb_ligands"] for v in requests.values())
print(f"\nSubmitted {len(requests)} jobs, {total_ligands} total ligands")
print(f"Request IDs saved to {HISTORY_FILE}")

## 5. Monitor Completion

Poll all submitted jobs to check how many ligand predictions have completed.

In [ ]:
# Reload request IDs if resuming from a previous session
# with open(HISTORY_FILE, "r") as f:
#     requests = json.load(f)

completed_requests = 0
failed_requests = 0
pending_requests = 0
processing_requests = 0
starting_requests = 0
other_requests = 0

for k, v in tqdm(requests.items()):
    results = client.get_results(parent_request_id=v["parent_request_id"])
    time.sleep(0.1)
    for ligand_name, result in results.items():
        if result["status"] == "COMPLETED":
            completed_requests += 1
        elif result["status"] == "FAILED":
            failed_requests += 1
        elif result["status"] == "PENDING":
            pending_requests += 1
        elif result["status"] == "PROCESSING":
            processing_requests += 1
        elif result["status"] == "STARTING":
            starting_requests += 1
        else:
            print(result["status"])
            other_requests += 1

print(f"Pending:   {pending_requests}")
print(f"Starting:  {starting_requests}")
print(f"Processing:   {processing_requests}")
print(f"Completed: {completed_requests}")
print(f"Failed:    {failed_requests}")
print(f"Other:     {other_requests}")

## 6. Collect Results into a DataFrame

In [ ]:
rows = []
for k, v in tqdm(requests.items()):
    results = client.get_results(parent_request_id=v["parent_request_id"])
    time.sleep(0.1)
    for ligand_name, result in results.items():
        if result["status"] == "COMPLETED":
            row = copy.deepcopy(result["data"])
            row["ligand_name"] = ligand_name
            row["protein_name"] = k.split("-")[0]
            row["file"] = row["protein_name"] + "__" + ligand_name
            rows.append(row)

df_results = pd.DataFrame(rows)
print(f"Collected {len(df_results)} predictions")
df_results.head()

## 7. Load Experimental Ground Truth

Each protein folder contains a `metadata.json` with experimental binding free energies (`exp_dG_kcal_mol`) per ligand.

In [ ]:
data_csv = []
for file in glob.glob("data/benchmark/*/metadata.json"):
    with open(file, "r") as f:
        data = json.load(f)
    entry = data["entry"]
    for lig_name, lig_data in data["ligands"].items():
        filename = f"{entry}__{lig_name}".replace(".", "_")
        data_csv.append({"file": filename, "exp_dG": lig_data["exp_dG_kcal_mol"]})

df_true = pd.DataFrame(data_csv)
print(f"Ground truth: {len(df_true)} entries")
df_true.head()

## 8. Merge Predictions with Ground Truth and Evaluate

In [ ]:
df_final = df_results.merge(df_true, on="file", how="inner")
df_clean = df_final.dropna(subset=["exp_dG"])
df_clean["pIC50"] = -df_clean["exp_dG"]
print(f"Matched predictions: {len(df_final)} (with exp. data: {len(df_clean)})")
df_final.info()

In [ ]:
s = spearmanr(df_clean["pIC50"].values, df_clean["affinity_pIC50_pred_value"].values)
p = pearsonr(df_clean["pIC50"].values, df_clean["affinity_pIC50_pred_value"].values)

print(f"Spearman r = {s.statistic:.4f}  (p = {s.pvalue:.2e})")
print(f"Pearson  r = {p.statistic:.4f}  (p = {p.pvalue:.2e})")
print(f"N = {len(df_clean)}")

## 9. Save Results

In [ ]:
output_file = f"df_results_{RUN_NAME}.csv"
df_final.to_csv(output_file, index=False)
print(f"Saved {len(df_final)} results to {output_file}")